# Harris County GraphSAGE block-group topology test

Both models use the refreshed 2021 filing labels and the original saved property splits. The geography-only arm adds bidirectional property–block-group edges and gives every block group the same constant feature. It can learn from properties sharing a block group. The experiment uses no ACS values or filing totals as inputs. Run with PyTorch, PyTorch Geometric and a supported neighbor sampling backend, pandas, pyarrow, NumPy, and scikit-learn. In Colab mount Drive and set BASE to the folder with the saved graph files, refreshed labels and block-group mapping.

## Test results

~~~text
           run  seed  average_precision  roc_auc
       updated    42           0.249495 0.788018
geography_only    42           0.258462 0.795682
       updated    43           0.247484 0.792837
geography_only    43           0.257297 0.793963
       updated    44           0.250453 0.792232
geography_only    44           0.262189 0.798798
~~~

~~~text
               average_precision             roc_auc          
                            mean       std      mean       std
run                                                           
geography_only          0.259316  0.002556  0.796148  0.002451
updated                 0.249144  0.001515  0.791029  0.002625
~~~

The spread across these three seeds describes run-to-run variation. The graph uses the original property split, which allows properties from the same block group across splits.

In [ ]:
from pathlib import Path

BASE = Path("/content/drive/MyDrive/GNN")
REFRESHED_LABELS = BASE / "harris_2021_refreshed_property_labels.parquet"

BLOCK_GROUP_MAP = BASE / "harris_2021_property_block_groups.parquet"
RUN = ["updated","geography_only"]
EPOCHS = 10


In [ ]:
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from sklearn.metrics import average_precision_score, roc_auc_score
from torch_geometric.data import HeteroData
from torch_geometric.loader import NeighborLoader
from torch_geometric.nn import HeteroConv, SAGEConv

property_node_table = pd.read_parquet(BASE / "property_node_table_2021.parquet")
owner_node_table = pd.read_parquet(BASE / "owner_node_table_2021.parquet")
edges_2021 = pd.read_parquet(BASE / "gnn_edges_2021.parquet")
train_idx, val_idx, test_idx = [np.load(BASE / f"{split}_idx_2021.npy")
                                for split in ("train", "val", "test")]
for table, key in [(property_node_table, "property_node_id"), (owner_node_table, "owner_node_id")]:
    assert np.array_equal(table[key].to_numpy(), np.arange(len(table))), f"Unexpected {key} order"
indices = np.concatenate([train_idx,val_idx,test_idx])
assert len(np.unique(indices)) == len(indices)
assert property_node_table.iloc[indices].broader_rental.astype(bool).all()

def account_id(values):
    clean = values.astype("string").str.strip()
    return clean.str.zfill(13).where(clean.str.fullmatch(r"[0-9]{1,13}", na=False))

property_accounts = account_id(property_node_table["acct"])
label_table = pd.read_parquet(REFRESHED_LABELS)
label_table["acct"] = account_id(label_table["acct"])
assert property_accounts.is_unique and label_table.acct.is_unique and label_table.acct.notna().all()
old_labels = property_node_table["any_filing_2021"].astype("int8").to_numpy()
new_labels = property_accounts.map(label_table.set_index("acct")["any_filing_2021"]).fillna(0).astype("int8").to_numpy()
print("Refreshed positive accounts missing from original graph:",
      int((~label_table.acct.isin(property_accounts)).sum()))
for split, idx in zip(("train","validation","test"),(train_idx,val_idx,test_idx)):
    old, new = old_labels[idx],new_labels[idx]
    print(split, "properties",len(idx),"old positive",int(old.sum()),"new positive",int(new.sum()),
          "old only",int(((old==1)&(new==0)).sum()),"new only",int(((old==0)&(new==1)).sum()))

In [ ]:
draft_feature_cols = [
    "exterior_wall_type_count",
    "has_central_hvac",
    "mean_condition_score",
    "mean_grade_adj",
    "has_slab_foundation",
    "full_baths",
    "mean_cdu_adj",
    "has_pier_foundation",
    "bedrooms",
    "foundation_type_count",
    "total_rooms",
    "fireplace_count",
    "has_cracked_slab",
    "has_pool_spa",
    "has_foundation_repair",
    "has_solar",
    "has_pool",
    "has_porch",
    "has_shed",
    "has_carport",
    "has_outdoor_kitchen",
    "has_garage",
    "half_baths"
]

# -----------------------------
# PROPERTY FEATURES
# -----------------------------

X_property = (
    property_node_table[draft_feature_cols]
    .apply(pd.to_numeric, errors="coerce")
    .copy()
)

# Missingness indicators
missing_indicator_cols = [
    c for c in draft_feature_cols
    if X_property[c].isna().any()
]

for c in missing_indicator_cols:
    X_property[f"{c}_missing"] = (
        X_property[c].isna().astype(np.float32)
    )

# IMPORTANT: medians from TRAINING target properties only
train_medians = X_property.iloc[train_idx][draft_feature_cols].median()

for c in draft_feature_cols:
    X_property[c] = X_property[c].fillna(train_medians[c])

X_property = X_property.astype(np.float32)

# Convert before scaling
x_prop = torch.tensor(
    X_property.to_numpy(),
    dtype=torch.float32
)

# Scale using training nodes only
prop_mean = x_prop[train_idx].mean(dim=0)
prop_std = x_prop[train_idx].std(dim=0)
prop_std[prop_std == 0] = 1.0

x_prop = (x_prop - prop_mean) / prop_std


# -----------------------------
# OWNER FEATURES
# -----------------------------

x_owner = torch.tensor(
    owner_node_table[
        ["log_portfolio_size", "has_corporate_owner"]
    ].to_numpy(dtype=np.float32),
    dtype=torch.float32
)

# Scale log portfolio only
owner_mean = x_owner[:, 0].mean()
owner_std = x_owner[:, 0].std()

x_owner[:, 0] = (
    x_owner[:, 0] - owner_mean
) / owner_std

print("Property x:", x_prop.shape)
print("Owner x:", x_owner.shape)
print("Property finite:", torch.isfinite(x_prop).all().item())
print("Owner finite:", torch.isfinite(x_owner).all().item())

In [ ]:

mapping = pd.read_parquet(BLOCK_GROUP_MAP, columns=["property_node_id", "acct", "block_group_geoid"])
assert np.array_equal(mapping.property_node_id.to_numpy(), property_node_table.property_node_id.to_numpy())
assert account_id(mapping.acct).equals(property_accounts)
geoid = mapping.block_group_geoid
groups = pd.Index(geoid.dropna().unique()).sort_values()
block_idx = groups.get_indexer(geoid)
property_idx = np.flatnonzero(block_idx >= 0)
block_edges = torch.as_tensor(np.stack([property_idx,block_idx[property_idx]]),dtype=torch.long)
block_features = torch.ones((len(groups),1),dtype=torch.float32)
print("Block groups",len(groups),"property-to-block-group edges",len(property_idx))


In [ ]:
owns = ("owner","owns","property")
owned_by = ("property","owned_by","owner")
in_group = ("property","in_block_group","block_group")
contains = ("block_group","contains","property")

def build_graph(arm):
    graph = HeteroData()
    graph["property"].x = torch.cat([x_prop,context[0]],dim=1) if arm=="acs_property" else x_prop
    labels = old_labels if arm=="original" else new_labels
    graph["property"].y = torch.tensor(labels,dtype=torch.float32)
    graph["property"].common_y = torch.tensor(new_labels,dtype=torch.float32)
    graph["owner"].x = x_owner
    ownership = torch.tensor(edges_2021[["owner_node_id","property_node_id"]].to_numpy().T,dtype=torch.long)
    graph[owns].edge_index = ownership
    graph[owned_by].edge_index = ownership.flip(0)
    if arm=="geography_only":
        graph["block_group"].x = block_features
        graph[in_group].edge_index = block_edges
        graph[contains].edge_index = block_edges.flip(0)
    if arm=="acs_node":
        graph["block_group"].x = context[1]
        graph[in_group].edge_index = context[2]
        graph[contains].edge_index = context[2].flip(0)
    return graph

class EvictionSAGE(torch.nn.Module):
    def __init__(self,relations,width=64):
        super().__init__()
        self.conv1 = HeteroConv({rel:SAGEConv((-1,-1),width) for rel in relations},aggr="sum")
        self.conv2 = HeteroConv({rel:SAGEConv((width,width),width) for rel in relations},aggr="sum")
        self.classifier = torch.nn.Linear(width,1)

    def forward(self,batch):
        hidden = {key:F.relu(value) for key,value in self.conv1(batch.x_dict,batch.edge_index_dict).items()}
        return self.classifier(F.relu(self.conv2(hidden,batch.edge_index_dict)["property"])).squeeze(-1)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

@torch.no_grad()
def evaluate(model,loader):
    model.eval()
    collected = []
    for batch in loader:
        batch = batch.to(device)
        n = batch["property"].batch_size
        collected.append((batch["property"].y[:n].cpu().numpy(),
                          batch["property"].common_y[:n].cpu().numpy(),
                          model(batch)[:n].sigmoid().cpu().numpy()))
    target,common,score = (np.concatenate(columns) for columns in zip(*collected))
    return dict(average_precision=average_precision_score(target,score),
                roc_auc=roc_auc_score(target,score),
                ap_common_updated_labels=average_precision_score(common,score),
                roc_auc_common_updated_labels=roc_auc_score(common,score),
                positive_prevalence=float(target.mean()))

def experiment(arm, seed):
    torch.manual_seed(seed)
    graph = build_graph(arm)
    relations = graph.edge_types
    def loader(idx,batch_size,shuffle):
        return NeighborLoader(graph,input_nodes=("property",torch.as_tensor(idx)),
                              num_neighbors={rel:[5,10] for rel in relations},
                              batch_size=batch_size,shuffle=shuffle,num_workers=0)
    train,val,test = loader(train_idx,2048,True),loader(val_idx,4096,False),loader(test_idx,4096,False)
    model = EvictionSAGE(relations).to(device)
    optimizer = torch.optim.Adam(model.parameters(),lr=0.001,weight_decay=1e-4)
    positive = graph["property"].y[train_idx].sum().item()
    if positive == 0:
        raise ValueError("No positive training properties in this label set.")
    criterion = torch.nn.BCEWithLogitsLoss(
        pos_weight=torch.tensor([(len(train_idx)-positive)/positive],device=device))
    best_ap = -1
    for epoch in range(EPOCHS):
        model.train()
        for batch in train:
            batch = batch.to(device)
            n = batch["property"].batch_size
            optimizer.zero_grad()
            loss = criterion(model(batch)[:n],batch["property"].y[:n])
            loss.backward()
            optimizer.step()
        ap = evaluate(model,val)["average_precision"]
        print(arm,epoch+1,"validation average precision",round(ap,4))
        if ap > best_ap:
            best_ap = ap
            best_state = {key:value.detach().cpu().clone() for key,value in model.state_dict().items()}
    model.load_state_dict(best_state)
    return {"run":arm,"seed":seed,**evaluate(model,test)}



In [ ]:
results = pd.DataFrame([experiment(arm,seed) for seed in [42,43,44] for arm in RUN])
display(results)